# GNSS-Denied Navigation — Cross-View Matching

Train a model to match drone camera views against satellite imagery for GPS-free positioning.

**Pipeline:** EuroSAT satellite images → synthetic drone views → contrastive learning → feature extractor

In [ ]:
!pip install -q datasets timm

import os, cv2, glob, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from datasets import load_dataset

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/DroneCV/gnss_denied'
DATA_DIR = '/content/data/crossview_pairs'
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{DATA_DIR}/satellite', exist_ok=True)
os.makedirs(f'{DATA_DIR}/drone', exist_ok=True)
print('Setup complete')

## Step 1: Generate Training Data from EuroSAT

In [ ]:
print('Loading EuroSAT satellite images from HuggingFace...')
ds = load_dataset('blanchon/EuroSAT_RGB', split='train[:2000]')
print(f'Loaded {len(ds)} images')

np.random.seed(42)
for i, sample in enumerate(ds):
    img = np.array(sample['image'])
    sat = cv2.resize(img, (256, 256))
    cv2.imwrite(f'{DATA_DIR}/satellite/{i:04d}.jpg', cv2.cvtColor(sat, cv2.COLOR_RGB2BGR))
    # Simulate drone view: random crop + rotation + blur
    crop = int(256 * np.random.uniform(0.5, 0.85))
    x, y = np.random.randint(0, 256-crop), np.random.randint(0, 256-crop)
    drone = cv2.resize(sat[y:y+crop, x:x+crop], (256, 256))
    M = cv2.getRotationMatrix2D((128,128), np.random.uniform(-25,25), 1.0)
    drone = cv2.warpAffine(drone, M, (256,256))
    drone = np.clip(drone*np.random.uniform(0.8,1.2) + np.random.uniform(-15,15), 0, 255).astype(np.uint8)
    cv2.imwrite(f'{DATA_DIR}/drone/{i:04d}.jpg', cv2.cvtColor(drone, cv2.COLOR_RGB2BGR))

print(f'Generated {len(ds)} satellite↔drone pairs')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(5):
    idx = i * 200
    axes[0,i].imshow(cv2.cvtColor(cv2.imread(f'{DATA_DIR}/satellite/{idx:04d}.jpg'), cv2.COLOR_BGR2RGB))
    axes[0,i].set_title(f'Satellite {idx}'); axes[0,i].axis('off')
    axes[1,i].imshow(cv2.cvtColor(cv2.imread(f'{DATA_DIR}/drone/{idx:04d}.jpg'), cv2.COLOR_BGR2RGB))
    axes[1,i].set_title(f'Drone {idx}'); axes[1,i].axis('off')
plt.suptitle('Training Pairs: Satellite (top) vs Simulated Drone (bottom)')
plt.tight_layout(); plt.show()

## Step 2: Model + Dataset

In [ ]:
class CrossViewEncoder(nn.Module):
    def __init__(self, backbone='resnet50', embed_dim=512):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0)
        self.embed = nn.Sequential(
            nn.Linear(self.backbone.num_features, embed_dim),
            nn.BatchNorm1d(embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, embed_dim))
    def forward(self, x):
        return nn.functional.normalize(self.embed(self.backbone(x)), p=2, dim=1)

class TripletDataset(Dataset):
    def __init__(self, data_dir):
        self.sat = sorted(glob.glob(f'{data_dir}/satellite/*.jpg'))
        self.drone = sorted(glob.glob(f'{data_dir}/drone/*.jpg'))
        self.n = min(len(self.sat), len(self.drone))
        self.transform = transforms.Compose([
            transforms.Resize((256,256)), transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __len__(self): return self.n
    def __getitem__(self, idx):
        drone = self.transform(Image.open(self.drone[idx]).convert('RGB'))
        sat_pos = self.transform(Image.open(self.sat[idx]).convert('RGB'))
        neg = random.choice([j for j in range(self.n) if j != idx])
        sat_neg = self.transform(Image.open(self.sat[neg]).convert('RGB'))
        return drone, sat_pos, sat_neg

model = CrossViewEncoder().cuda()
dataset = TripletDataset(DATA_DIR)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Dataset: {len(dataset)} pairs, {len(loader)} batches/epoch')

## Step 3: Train (Triplet Loss)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
criterion = nn.TripletMarginLoss(margin=0.3)

for epoch in range(15):
    model.train()
    losses = []
    for drone, sat_pos, sat_neg in loader:
        drone, sat_pos, sat_neg = drone.cuda(), sat_pos.cuda(), sat_neg.cuda()
        loss = criterion(model(drone), model(sat_pos), model(sat_neg))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    print(f'Epoch {epoch+1}/15 — Loss: {np.mean(losses):.4f}')

torch.save(model.state_dict(), f'{WORK_DIR}/models/crossview_resnet50_512d.pth')
print(f'\nModel saved to Drive')

## Step 4: Evaluate — Retrieval Accuracy

In [ ]:
model.eval()
eval_tf = transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# Embed all satellite tiles as gallery
sat_files = sorted(glob.glob(f'{DATA_DIR}/satellite/*.jpg'))
drone_files = sorted(glob.glob(f'{DATA_DIR}/drone/*.jpg'))

def embed_batch(files):
    embs = []
    for i in range(0, len(files), 64):
        batch = torch.stack([eval_tf(Image.open(f).convert('RGB')) for f in files[i:i+64]]).cuda()
        with torch.no_grad(): embs.append(model(batch).cpu().numpy())
    return np.vstack(embs)

print('Embedding gallery + queries...')
sat_emb = embed_batch(sat_files)
drone_emb = embed_batch(drone_files[:500])

# Retrieval: for each drone image, find correct satellite match
sims = drone_emb @ sat_emb.T
top1 = sum(np.argmax(sims[i]) == i for i in range(len(drone_emb)))
top5 = sum(i in np.argsort(-sims[i])[:5] for i in range(len(drone_emb)))
n = len(drone_emb)
print(f'\nRetrieval (n={n}):')
print(f'  Top-1: {top1/n*100:.1f}%')
print(f'  Top-5: {top5/n*100:.1f}%')

## Step 5: Test on Custom Site (upload your own satellite image)

Upload a stitched satellite image to test localization on a real site.

In [ ]:
from google.colab import files
print('Upload a satellite reference image (e.g., jorvas_satellite_z18_stitched.jpg):')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    ref = cv2.imread(fname)
    print(f'Reference: {ref.shape}')
    
    # Split into 256x256 tiles and embed
    h, w = ref.shape[:2]
    tiles, centers = [], []
    for row in range(0, h-255, 128):
        for col in range(0, w-255, 128):
            tile = ref[row:row+256, col:col+256]
            t = eval_tf(Image.fromarray(cv2.cvtColor(tile, cv2.COLOR_BGR2RGB))).unsqueeze(0).cuda()
            with torch.no_grad(): tiles.append(model(t).cpu().numpy().flatten())
            centers.append((col+128, row+128))
    tile_emb = np.array(tiles)
    print(f'Embedded {len(tiles)} reference tiles')
    
    # Simulate drone view from center
    cy, cx = h//2, w//2
    drone_crop = ref[cy-100:cy+100, cx-100:cx+100]
    drone_crop = cv2.resize(drone_crop, (256,256))
    M = cv2.getRotationMatrix2D((128,128), 12, 1.0)
    drone_crop = cv2.warpAffine(drone_crop, M, (256,256))
    d = eval_tf(Image.fromarray(cv2.cvtColor(drone_crop, cv2.COLOR_BGR2RGB))).unsqueeze(0).cuda()
    with torch.no_grad(): d_emb = model(d).cpu().numpy().flatten()
    
    # Match
    sims = tile_emb @ d_emb
    best = np.argmax(sims)
    est = centers[best]
    true = (cx, cy)
    err_px = np.sqrt((est[0]-true[0])**2 + (est[1]-true[1])**2)
    print(f'Estimated: {est}, True: {true}, Error: {err_px:.0f}px')
    print(f'At GSD 0.30 m/px → position error: {err_px*0.3:.1f}m')
else:
    print('No file uploaded — skipping custom test')